# Identificación y autoría
**Speaker:** Eduardo Osorio Venegas  
**Apoyo de inteligencia artificial generativa:** ChatGPT (OpenAI).  
Material educativo con apoyo generativo en las celdas Markdown; requiere revisión y validación en Fabric.

# NB_AIFUNCTIONS_CONTROL_PANDAS
Versión Pandas de AI Functions en Microsoft Fabric. Conserva las cinco entradas y la auditoría original. Ejecutar en Fabric con `synapse.ml.aifunc` disponible. En un notebook Python puro instalar los wheels oficiales de SynapseML si faltan; en Runtime 1.3 PySpark normalmente ya están disponibles. [Requisitos](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/overview).

# Configurar Modelo

In [1]:
import pandas as pd
import synapse.ml.aifunc as aifunc
from datetime import datetime, timezone
from IPython.display import display
import uuid
import inspect

MODEL_NAME = "gpt-5-mini"
REASONING_EFFORT = "low"  # Comparar con minimal si el endpoint lo admite.
VERBOSITY = "low"
PERSIST_AUDIT = False  # Activar únicamente con un Lakehouse asociado.
TABLE_PATH = "/lakehouse/default/Tables/dbo/audit_ai_functions"

aifunc.default_conf.model_deployment_name = MODEL_NAME
for key, value in {
    "reasoning_effort": REASONING_EFFORT,
    "verbosity": VERBOSITY,
    "progress_bar_mode": "stats",
    "concurrency": 5,
}.items():
    if hasattr(aifunc.default_conf, key):
        setattr(aifunc.default_conf, key, value)
        print(f"{key}: {value}")
    else:
        print(f"No disponible en esta versión: {key}")
print(f"Modelo configurado: {MODEL_NAME}")

StatementMeta(, a6ff1757-0b7f-45bb-989e-a830fcfa8025, 3, Finished, Available, Finished, False)

reasoning_effort: low
verbosity: low
progress_bar_mode: stats
concurrency: 5
Modelo configurado: gpt-5-mini


# Registros ficticios

In [2]:

data = [
    (
        1,
        "Microsoft Fabric es una plataforma SaaS de analítica "
        "que integra ingeniería de datos, ciencia de datos y BI."
    ),
    (
        2,
        "OneLake permite centralizar los datos de una organización "
        "en una estructura de almacenamiento unificada."
    ),
    (
        3,
        "Delta Lake permite almacenar datos mediante transacciones "
        "ACID y control de versiones."
    ),
    (
        4,
        "Microsoft Purview permite implementar capacidades "
        "de gobierno, catálogo y clasificación de datos."
    ),
    (
        5,
        "PySpark permite procesar grandes volúmenes de datos "
        "mediante procesamiento distribuido."
    )
]

df = pd.DataFrame(data, columns=["id", "descripcion"])

display(df)


StatementMeta(, a6ff1757-0b7f-45bb-989e-a830fcfa8025, 4, Finished, Available, Finished, False)

,id,descripcion
0,1,Microsoft Fabric es una plataforma SaaS de ana...
1,2,OneLake permite centralizar los datos de una o...
2,3,Delta Lake permite almacenar datos mediante tr...
3,4,Microsoft Purview permite implementar capacida...
4,5,PySpark permite procesar grandes volúmenes de ...


## Diagnóstico de la API instalada

In [3]:
print("Pandas:", pd.__version__)
print("generate_response:", inspect.signature(df.ai.generate_response))

StatementMeta(, a6ff1757-0b7f-45bb-989e-a830fcfa8025, 5, Finished, Available, Finished, False)

Pandas: 2.1.4
generate_response: (prompt: str | collections.abc.Callable[..., str], *, is_prompt_template: bool = False, pass_row: bool = False, conf: Optional[synapse.ml.aifunc.core.Conf] = None, response_format: Union[synapse.ml.openaishim._types.NotGiven, NoneType, str, dict, type] = NOT_GIVEN, column_type_dict: Optional[dict[str, Literal['text', 'path']]] = None) -> pandas.core.series.Series


# Ejecutar AI Functions

* Explicación con ia generativa

La API Pandas devuelve una Series. Capturamos sus estadísticas antes de asignarla al DataFrame. Reejecutar esta celda genera nuevas solicitudes al modelo.

In [4]:
run_id = str(uuid.uuid4())
execution_timestamp = datetime.now(timezone.utc).isoformat()
respuestas = df.ai.generate_response(
    prompt=("Explica el siguiente concepto tecnológico "
            "en un máximo de 40 palabras: {descripcion}"),
    is_prompt_template=True
)
# Captura única de las métricas de esta ejecución.
stats_df = respuestas.ai.stats.copy(deep=True)
resultado = df.copy()
resultado["respuesta"] = respuestas
display(resultado)

StatementMeta(, a6ff1757-0b7f-45bb-989e-a830fcfa8025, 6, Finished, Available, Finished, False)

ai.generate_response: 100%|██████████| 5/5 [00:05<00:00,  1.10s/it, cached=0, in=246, out=1.52k, CU-s=102.40]                            


,id,descripcion,respuesta
0,1,Microsoft Fabric es una plataforma SaaS de ana...,Microsoft Fabric es una plataforma SaaS de aná...
1,2,OneLake permite centralizar los datos de una o...,OneLake centraliza los datos de una organizaci...
2,3,Delta Lake permite almacenar datos mediante tr...,Delta Lake es un almacenamiento de datos que a...
3,4,Microsoft Purview permite implementar capacida...,Microsoft Purview es una plataforma de gobiern...
4,5,PySpark permite procesar grandes volúmenes de ...,PySpark es la interfaz Python de Apache Spark ...


# Validar el consumo de tokens

###### **ai.stats** url: https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pandas/configuration

###### **Diccionario de campos**

* num_successful:	Filas procesadas correctamente
* num_exceptions:	Filas con excepciones
* num_unevaluated:	Filas no evaluadas
* num_harmful:	Filas bloqueadas por filtros
* input_tokens:	Tokens de entrada
* cached_tokens:	Tokens de entrada en caché
* output_tokens:	Tokens generados
* reasoning_tokens:	Tokens de razonamiento
* model: Modelo utilizado

In [5]:
display(stats_df)
if not isinstance(stats_df, pd.DataFrame) or len(stats_df) != 1:
    raise RuntimeError("Se esperaba una fila de estadísticas para una operación y un modelo; revisar stats_df.")
stats = stats_df.iloc[0].to_dict()
for key, value in stats.items():
    print(f"{key}: {value}")

StatementMeta(, a6ff1757-0b7f-45bb-989e-a830fcfa8025, 7, Finished, Available, Finished, False)

,num_successful,num_exceptions,num_unevaluated,num_harmful,cached_tokens,input_tokens,output_tokens,reasoning_tokens,client_type,input_types,model
0,5,0,0,0,0,246,1520,1152,fabric_llm_endpoint,{'text': 5},gpt-5-mini


num_successful: 5
num_exceptions: 0
num_unevaluated: 0
num_harmful: 0
cached_tokens: 0
input_tokens: 246
output_tokens: 1520
reasoning_tokens: 1152
client_type: fabric_llm_endpoint
input_types: {'text': 5}
model: gpt-5-mini


# Cálculo aproximado de capacidad

**Referencia de facturación:** https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/billing?tabs=pandas

Microsoft Fabric AI Functions permite estimar el consumo de capacidad asociado a las solicitudes realizadas a los modelos de inteligencia artificial. Este consumo se expresa en **CU-segundos (Capacity Units por segundo)** y depende de la cantidad y el tipo de tokens procesados durante cada ejecución.

Para realizar esta estimación se utiliza la siguiente fórmula, considerando las tarifas de `gpt-5-mini`:

$$
CU_s = \frac{(I-C)\times 8.40 + C\times 0.84 + O\times 67.23}{1000}
$$

**Variables de la fórmula:**

- **I (Input Tokens):** cantidad total de tokens de entrada enviados al modelo.
- **C (Cached Input Tokens):** cantidad de tokens de entrada reutilizados desde la caché.
- **O (Output Tokens):** cantidad de tokens generados como respuesta por el modelo.

**Descripción del cálculo:**

El cálculo separa los tokens de entrada en dos categorías: tokens procesados normalmente y tokens recuperados desde la caché. Esto permite aplicar una tarifa diferenciada a cada categoría.

1. **Entrada normal:** se restan los tokens en caché del total de tokens de entrada `(I - C)` y se multiplican por la tarifa de 8,40 CU-segundos por cada 1.000 tokens.
2. **Entrada en caché:** los tokens reutilizados desde la caché `(C)` se multiplican por la tarifa reducida de 0,84 CU-segundos por cada 1.000 tokens.
3. **Salida del modelo:** los tokens generados por el modelo `(O)` se multiplican por la tarifa de 67,23 CU-segundos por cada 1.000 tokens.
4. **Consumo total:** se suman los tres componentes y se divide el resultado por 1.000 para obtener el consumo estimado expresado en CU-segundos.

**Consideraciones:**

- La estimación utiliza las tarifas correspondientes al modelo `gpt-5-mini`.
- El consumo puede variar según el modelo utilizado y las tarifas de facturación vigentes.
- Los tokens de entrada en caché deben estar incluidos dentro del total de tokens de entrada, no contabilizarse como una categoría adicional.
- El resultado representa una estimación del consumo asociado al procesamiento de tokens, no necesariamente el consumo total de capacidad de Microsoft Fabric de todos los componentes de la ejecución.


Los tokens de razonamiento se muestran por separado, pero no se suman otra vez a la salida total.


In [6]:
def metric(name):
    value = stats.get(name)
    return None if value is None or pd.isna(value) else int(value)

input_tokens = metric("input_tokens")
cached_tokens = metric("cached_tokens")
output_tokens = metric("output_tokens")
reasoning_tokens = metric("reasoning_tokens")
if any(x is None for x in (input_tokens, cached_tokens, output_tokens)):
    raise RuntimeError("Faltan métricas necesarias para estimar CU.")
if not 0 <= cached_tokens <= input_tokens or output_tokens < 0:
    raise ValueError("Métricas de tokens inconsistentes")
INPUT_RATE, CACHED_RATE, OUTPUT_RATE = 8.40, 0.84, 67.23
reported = stats.get("model")
if pd.isna(reported) or reported not in ("gpt-5-mini", "gpt-5-mini-2025-08-07"):
    raise ValueError(f"Tarifas no verificadas para modelo reportado: {reported}")
cu_input = (input_tokens - cached_tokens) * INPUT_RATE / 1000
cu_cache = cached_tokens * CACHED_RATE / 1000
cu_output = output_tokens * OUTPUT_RATE / 1000
estimated_cu_seconds = cu_input + cu_cache + cu_output
print(f"Tokens totales: {input_tokens + output_tokens:,}")
print(f"Tokens de razonamiento: {reasoning_tokens}")
print(f"CU-s estimados: {estimated_cu_seconds:.4f}")
display(pd.DataFrame([{"cu_input": cu_input, "cu_cache": cu_cache,
    "cu_output": cu_output, "cu_total": estimated_cu_seconds,
    "reasoning_pct": 100 * reasoning_tokens / output_tokens if reasoning_tokens is not None and output_tokens else None}]))

StatementMeta(, a6ff1757-0b7f-45bb-989e-a830fcfa8025, 8, Finished, Available, Finished, False)

Tokens totales: 1,766
Tokens de razonamiento: 1152
CU-s estimados: 104.2560


,cu_input,cu_cache,cu_output,cu_total,reasoning_pct
0,2.0664,0.0,102.1896,104.256,75.789474


# Crear tabla de auditoria

* run_id:	Identificador único de ejecución
* execution_timestamp:	Fecha y hora UTC
* notebook_name:	Notebook ejecutado
* operation:	Función AI utilizada
* model_configured:	Modelo configurado
* model_reported:	Modelo informado por Fabric
* num_input_rows:	Filas enviadas
* num_successful:	Procesamientos exitosos
* num_exceptions:	Excepciones
* num_unevaluated:	Filas no evaluadas
* num_harmful:	Filas bloqueadas
* input_tokens:	Tokens de entrada
* cached_tokens:	Tokens en caché
* output_tokens:	Tokens de salida
* reasoning_tokens:	Tokens de razonamiento
* estimated_cu_seconds:	CU-segundos estimados
* client_type:	Tipo de endpoint
* execution_status:	Estado de la ejecución

In [7]:
num_input_rows = len(df)
successful = metric("num_successful")
exceptions = metric("num_exceptions")
unevaluated = metric("num_unevaluated")
harmful = metric("num_harmful")
if successful == num_input_rows and not any(x for x in (exceptions, unevaluated, harmful)):
    execution_status = "SUCCESS"
elif successful == 0 and (exceptions or harmful):
    execution_status = "FAILED"
elif exceptions or harmful:
    execution_status = "PARTIAL"
else:
    execution_status = "INCOMPLETE"
record = dict(run_id=run_id, execution_timestamp=execution_timestamp,
    notebook_name="NB_AIFUNCTIONS_CONTROL_PANDAS", operation="generate_response",
    model_configured=MODEL_NAME, model_reported=reported,
    num_input_rows=num_input_rows, num_successful=successful,
    num_exceptions=exceptions, num_unevaluated=unevaluated, num_harmful=harmful,
    input_tokens=input_tokens, cached_tokens=cached_tokens, output_tokens=output_tokens,
    reasoning_tokens=reasoning_tokens, estimated_cu_seconds=estimated_cu_seconds,
    client_type=stats.get("client_type"), execution_status=execution_status)
audit_df = pd.DataFrame([record])
for field in ["num_input_rows", "num_successful", "num_exceptions", "num_unevaluated", "num_harmful", "input_tokens", "cached_tokens", "output_tokens", "reasoning_tokens"]:
    audit_df[field] = audit_df[field].astype("Int64")
for field in audit_df.columns:
    if audit_df[field].dtype == object:
        audit_df[field] = audit_df[field].astype("string")
display(audit_df)

StatementMeta(, a6ff1757-0b7f-45bb-989e-a830fcfa8025, 9, Finished, Available, Finished, False)

,run_id,execution_timestamp,notebook_name,operation,model_configured,model_reported,num_input_rows,num_successful,num_exceptions,num_unevaluated,num_harmful,input_tokens,cached_tokens,output_tokens,reasoning_tokens,estimated_cu_seconds,client_type,execution_status
0,29291d89-c948-493d-b183-a2c3b676c9be,2026-11-25T04:51:59.349911+00:00,NB_AIFUNCTIONS_CONTROL_PANDAS,generate_response,gpt-5-mini,gpt-5-mini,5,5,0,0,0,246,0,1520,1152,104.256,fabric_llm_endpoint,SUCCESS


# Persistir tabla de auditoría
Sin Spark: usar delta-rs (`deltalake`) y PyArrow. Requiere un Lakehouse predeterminado con schemas habilitados y acceso de escritura. Si falta la biblioteca, instalar `deltalake` en el entorno. Se asume que existe la tabla Delta original `dbo.audit_ai_functions`, para no crear inadvertidamente una tabla sin registrar. La persistencia es opcional y conserva el append original; reejecutar esta celda puede duplicar el run_id.

In [ ]:
if PERSIST_AUDIT:
    import pyarrow as pa
    from deltalake import DeltaTable, write_deltalake
    target = DeltaTable(TABLE_PATH)
    arrow_table = pa.Table.from_pandas(audit_df, preserve_index=False)
    write_deltalake(target, arrow_table, mode="append")
    print(f"Auditoría guardada: {TABLE_PATH}")
else:
    print("Persistencia desactivada. audit_df permanece disponible en memoria.")